# Experiment: Protobuf Packet Investigation

Objective:
- Identify recurring packet/message patterns from local protobuf captures.
- Produce a quick, reproducible summary (message IDs + common field numbers) for reverse-engineering notes.


In [ ]:
# Setup: imports and reproducibility
from __future__ import annotations

from collections import Counter
from pathlib import Path
import re
import subprocess

SEED = 7  # Placeholder to keep experiment scaffolding deterministic where needed.


def find_repo_root(start: Path) -> Path:
    for candidate in (start, *start.parents):
        if (candidate / ".git").exists() and (candidate / "Source").exists():
            return candidate
    return start


REPO_ROOT = find_repo_root(Path.cwd().resolve())
PROTOC_PATH = REPO_ROOT / "Source" / "ThirdParty" / "protobuf-2.6.1rc1" / "tools" / "protoc.exe"
DUMP_DIR = REPO_ROOT / "Temp" / "ProtobufDump"
MESSAGE_TRACE_DIR = REPO_ROOT / "Temp" / "MessageId_Trace"

{
    "repo_root": str(REPO_ROOT),
    "protoc_exists": PROTOC_PATH.exists(),
    "dump_dir_exists": DUMP_DIR.exists(),
    "message_trace_exists": MESSAGE_TRACE_DIR.exists(),
}

## Plan

- Hypothesis: a small subset of fields and message IDs dominate captures and can be catalogued quickly.
- Variables to sweep: capture directory, extension filter (`.bin` / `.dat`), decode limit.
- Metrics to record: decoded file count, unique message names, top field frequencies.


In [ ]:
# Define helpers used by the baseline run

MESSAGE_ID_FILENAME_RE = re.compile(r"^.+?_([0-9A-Fa-fx]+)_.+?_.+?_(.+?)\.[^.]+$")


def decode_raw_file(path: Path, protoc_path: Path = PROTOC_PATH) -> str:
    if not protoc_path.exists():
        raise FileNotFoundError(f"protoc not found: {protoc_path}")

    result = subprocess.run(
        [str(protoc_path), "--decode_raw"],
        input=path.read_bytes(),
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        check=False,
    )
    return result.stdout.decode("utf-8", errors="replace")


def decode_raw_directory(
    dump_dir: Path = DUMP_DIR,
    patterns: tuple[str, ...] = ("*.bin", "*.dat"),
    limit: int = 20,
) -> dict[str, str]:
    files: list[Path] = []
    for pattern in patterns:
        files.extend(sorted(dump_dir.glob(pattern)))

    files = files[:limit]
    decoded: dict[str, str] = {}

    for file_path in files:
        decoded[file_path.name] = decode_raw_file(file_path)

    return decoded


def extract_message_ids(trace_dir: Path = MESSAGE_TRACE_DIR) -> dict[str, str]:
    # Adapted from Tools/Utilities/extract_unique_message_ids.py
    unique_ids: dict[str, str] = {}

    for file_path in trace_dir.rglob("*"):
        if not file_path.is_file():
            continue

        parts = file_path.name.split("_")
        if len(parts) >= 5:
            packet_id = parts[1]
            packet_name = parts[4].split(".")[0]
            unique_ids.setdefault(packet_name, packet_id)
            continue

        match = MESSAGE_ID_FILENAME_RE.match(file_path.name)
        if match:
            packet_id, packet_name = match.group(1), match.group(2)
            unique_ids.setdefault(packet_name, packet_id)

    return dict(sorted(unique_ids.items()))


def summarize_field_numbers(decoded_map: dict[str, str]) -> Counter[int]:
    field_counter: Counter[int] = Counter()
    for decoded_text in decoded_map.values():
        for line in decoded_text.splitlines():
            line = line.strip()
            if not line:
                continue
            head = line.split(":", 1)[0].strip()
            if head.isdigit():
                field_counter[int(head)] += 1
    return field_counter

## Results

- Baseline run below decodes up to 20 packet dumps and summarizes the most common fields.
- Keep outputs compact; move deep dives into follow-up cells as needed.
- Decision: if useful signals appear, promote logic into reusable scripts under `Tools/Utilities`.


In [ ]:
# Baseline execution and compact summary
decoded: dict[str, str] = {}
message_ids: dict[str, str] = {}
top_fields: list[tuple[int, int]] = []

if DUMP_DIR.exists() and PROTOC_PATH.exists():
    decoded = decode_raw_directory(DUMP_DIR, limit=20)
    top_fields = summarize_field_numbers(decoded).most_common(15)

if MESSAGE_TRACE_DIR.exists():
    message_ids = extract_message_ids(MESSAGE_TRACE_DIR)

result = {
    "decoded_files": len(decoded),
    "unique_message_names": len(message_ids),
    "top_fields": top_fields,
    "sample_message_ids": list(message_ids.items())[:25],
}

result

## Next steps

- Add one comparison cell for two captures of the same packet type (diff-style field presence).
- Export `sample_message_ids` and `top_fields` to CSV/Markdown for commit-friendly artifacts.
- Convert stable helpers into a shared Python module used by both scripts and this notebook.
